In [1]:
import sys
sys.path.append("..")

import gc
from src.datos import cargar_obt, agregar_features
from src.pipeline import preparar_datos, configurar_mlflow, tuning, SEMILLA

df = agregar_features(cargar_obt())
entrenamiento, prueba = preparar_datos(df)
del df
gc.collect()

configurar_mlflow()
print("Entrenamiento:", entrenamiento.shape)

Entrenamiento: (757018, 59)


In [2]:
from scipy.stats import loguniform, randint
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

hgb = HistGradientBoostingRegressor(random_state=SEMILLA)
espacio_hgb = {
    "learning_rate": loguniform(0.01, 0.3),
    "max_iter": randint(100, 600),
    "max_leaf_nodes": randint(15, 128),
    "min_samples_leaf": randint(20, 500),
    "l2_regularization": loguniform(1e-3, 10),
}

rf = RandomForestRegressor(n_jobs=2, random_state=SEMILLA)
espacio_rf = {
    "n_estimators": randint(50, 201),
    "max_depth": [10, 15, 20, 30, None],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_features": [0.3, 0.5, 0.7, 1.0],
    "max_samples": [0.05, 0.1, 0.15],
}

mlp = TransformedTargetRegressor(
    regressor=MLPRegressor(early_stopping=True, random_state=SEMILLA),
    transformer=StandardScaler(),
)
espacio_mlp = {
    "regressor__hidden_layer_sizes": [(50,), (100,), (64, 32), (100, 50), (128, 64)],
    "regressor__alpha": loguniform(1e-5, 1e-1),
    "regressor__learning_rate_init": loguniform(1e-4, 1e-2),
    "regressor__batch_size": [200, 500, 1000],
}

In [4]:
from sklearn.base import clone
from src.pipeline import evaluar_y_registrar

modelos = [
    (hgb, espacio_hgb, "hgb", "boosting"),
    (rf, espacio_rf, "random_forest", "bagging"),
    (mlp, espacio_mlp, "mlp", "red_neuronal"),
]
resultados_tuning = {}

for estimador, espacio, nombre, familia in modelos:
    try:
        tabla, mejor = tuning(estimador, espacio, nombre=nombre, familia=familia,
                              entrenamiento=entrenamiento)
        evaluar_y_registrar(clone(estimador).set_params(**mejor), nombre=f"tuning_{nombre}_mejor",
                            familia=familia, fase="tuning", entrenamiento=entrenamiento)
        resultados_tuning[nombre] = (tabla, mejor)
        print(f"=== {nombre} terminado ===")
    except Exception as e:
        print(f"=== {nombre} falló: {e} ===")

🏃 View run hgb_config_01 at: http://localhost:5000/#/experiments/1/runs/dbdbb371b78947f9a4717a9bc20f0894
🧪 View experiment at: http://localhost:5000/#/experiments/1
hgb config 1/30: WAPE 0.0905 | 1.4 min
🏃 View run hgb_config_02 at: http://localhost:5000/#/experiments/1/runs/5679591ffef6473282a8891ffc1ab7ab
🧪 View experiment at: http://localhost:5000/#/experiments/1
hgb config 2/30: WAPE 0.0878 | 2.7 min
🏃 View run hgb_config_03 at: http://localhost:5000/#/experiments/1/runs/55315d57134a44cc9cc992f40ea7cca3
🧪 View experiment at: http://localhost:5000/#/experiments/1
hgb config 3/30: WAPE 0.0921 | 1.1 min
🏃 View run hgb_config_04 at: http://localhost:5000/#/experiments/1/runs/32f196b190464987bfc25127ca5a85a8
🧪 View experiment at: http://localhost:5000/#/experiments/1
hgb config 4/30: WAPE 0.0869 | 2.7 min
🏃 View run hgb_config_05 at: http://localhost:5000/#/experiments/1/runs/95346253dfbc4e93a1cd9f38de676b1e
🧪 View experiment at: http://localhost:5000/#/experiments/1
hgb config 5/30: WA

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run tuning_hgb_mejor at: http://localhost:5000/#/experiments/1/runs/d6e81855a4cc42c9b9219b99d7e44a41
🧪 View experiment at: http://localhost:5000/#/experiments/1
=== hgb terminado ===
🏃 View run random_forest_config_01 at: http://localhost:5000/#/experiments/1/runs/7595a3e01af34d6580b7a82384c42caf
🧪 View experiment at: http://localhost:5000/#/experiments/1
random_forest config 1/30: WAPE 0.0885 | 1.7 min
🏃 View run random_forest_config_02 at: http://localhost:5000/#/experiments/1/runs/a04c2239d51f4e9cb286fd30f239e95d
🧪 View experiment at: http://localhost:5000/#/experiments/1
random_forest config 2/30: WAPE 0.0870 | 2.0 min
🏃 View run random_forest_config_03 at: http://localhost:5000/#/experiments/1/runs/f2e8e44b9ccf486a807e938084a7ac1d
🧪 View experiment at: http://localhost:5000/#/experiments/1
random_forest config 3/30: WAPE 0.0922 | 2.0 min
🏃 View run random_forest_config_04 at: http://localhost:5000/#/experiments/1/runs/965cb527a6aa4b3d817471e424046f35
🧪 View experiment at: h

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run tuning_random_forest_mejor at: http://localhost:5000/#/experiments/1/runs/2e9af5153c03402aa4e0f653be8d90bf
🧪 View experiment at: http://localhost:5000/#/experiments/1
=== random_forest terminado ===
🏃 View run mlp_config_01 at: http://localhost:5000/#/experiments/1/runs/f98fa88fb2cc4328a47d5bf7ac44a648
🧪 View experiment at: http://localhost:5000/#/experiments/1
mlp config 1/30: WAPE 0.1011 | 11.2 min
🏃 View run mlp_config_02 at: http://localhost:5000/#/experiments/1/runs/cb6331dae7714134a14a186a39f0a425
🧪 View experiment at: http://localhost:5000/#/experiments/1
mlp config 2/30: WAPE 0.0989 | 14.4 min
🏃 View run mlp_config_03 at: http://localhost:5000/#/experiments/1/runs/d1881aa234e7417eb6034377090e4c45
🧪 View experiment at: http://localhost:5000/#/experiments/1
mlp config 3/30: WAPE 0.1005 | 34.1 min
🏃 View run mlp_config_04 at: http://localhost:5000/#/experiments/1/runs/c1eb7c9a9d644f26a6a7cb2ddb27e55b
🧪 View experiment at: http://localhost:5000/#/experiments/1
mlp config

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run tuning_mlp_mejor at: http://localhost:5000/#/experiments/1/runs/1e4fb34a0ab64a1ea4d78c232d021b9a
🧪 View experiment at: http://localhost:5000/#/experiments/1
=== mlp terminado ===
